# BICA v0.2: независимая саморефлексия + zoom

Запускает ту же Qwen2.5-VL-3B-Instruct на шести **отладочных** изображениях. Сравнивает 4 baseline-режима и `reflect`, `reflect_zoom` на одинаковых входах. Итого 36 случаев, максимум 96 вызовов VLM. Не является доказательством качества.

Выполняйте ячейки по порядку в Colab с GPU; нужен ZIP `deliverables/bica_colab_bundle.zip`. Результаты сохраняются в Google Drive. После запуска нужен слепой человеческий скоринг.


In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.11.0 pillow==12.0.0

## После первой установки — перезапустите сеанс

После завершения предыдущей ячейки выберите «Среда выполнения → Перезапустить сеанс» (Runtime → Restart session). Это нужно, чтобы Colab загрузил установленные версии библиотек: Pillow может быть уже импортирован средой до установки.

После перезапуска продолжайте со следующей ячейки проверки GPU. Ячейку установки повторять не нужно, пока сохраняется эта виртуальная машина. Если Colab выделил новую машину, установите библиотеки снова и снова перезапустите сеанс.

Это обычный перезапуск сеанса, а не удаление среды.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from PIL import Image, ImageText
except ImportError as exc:
    raise RuntimeError('Pillow загружен некорректно после установки. Выберите Среда выполнения → Перезапустить сеанс и снова запустите ЭТУ ячейку, пропустив установку.') from exc
import PIL
print('Pillow:', PIL.__version__)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab не выделил GPU. Выберите T4 GPU в настройках runtime и подключитесь снова. Платить не требуется; при отсутствии квоты попробуйте позже.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM, GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('PyTorch:', torch.__version__)

## Сохранение результатов

Следующая ячейка запросит подключение Google Drive. Разрешение выдаётся Google в его интерфейсе. Результаты будут в My Drive/BICA_free. Завершённые случаи сохраняются отдельно; после разрыва runtime повторите ячейки, выбрав тот же архив и папку. Незавершённый случай запускается заново и учитывается в лимите.

In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')
SAVE = Path('/content/drive/MyDrive/BICA_free')
SAVE.mkdir(parents=True, exist_ok=True)
print('Результаты:', SAVE)

In [ ]:
import zipfile
import io

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Выберите только bica_colab_bundle.zip')
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.endswith('.zip'):
    raise ValueError('Нужен ZIP-архив проекта')
ROOT = Path('/content/bica')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for item in archive.infolist():
        target = (ROOT / item.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Небезопасный путь в архиве')
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
from illusion_agent.core import load_manifest
MANIFEST = ROOT / 'data/ambibench_debug/manifest.jsonl'
rows = load_manifest(MANIFEST)
print('Примеров:', len(rows), '| split:', sorted({r['split'] for r in rows}))
print('Условия:', sorted({c['id'] for r in rows for c in r['cues']}))
print('Эталонные ответы не загружаются в контекст модели.')

## Загрузка модели

Qwen2.5-VL-3B-Instruct — открытая модель для первого пилота. Веса загрузятся в Colab, не на Mac. Это несколько гигабайт и может занять время. Используем float16, greedy decoding и ограничение размера изображения. Модель не называется SOTA; второй моделью и опубликованными агентными методами проект нужно дополнить позже.

При нехватке видеопамяти остановите выполнение и сохраните текст ошибки. Не переходите автоматически на другую модель: это другой эксперимент.

In [ ]:
from illusion_agent.colab import QwenClient

revision_file = SAVE / 'qwen_model_revision.json'
revision = json.loads(revision_file.read_text())['revision'] if revision_file.exists() else None
client = QwenClient(revision=revision, max_tokens=768)
if not revision_file.exists():
    revision_file.write_text(json.dumps(client.metadata, indent=2))
print(json.dumps(client.metadata, indent=2))

In [ ]:
import importlib
import illusion_agent.core as core
import illusion_agent.colab as colab

# Обновить код после загрузки ZIP, сохранив уже загруженную модель client.
importlib.invalidate_caches()
importlib.reload(core)
importlib.reload(colab)
from illusion_agent.colab import run_pilot

SIX_MODES = ("direct", "repeat", "hypotheses", "evidence", "reflect", "reflect_zoom")

# Проверяем все шесть режимов: 6 случаев, 16 вызовов.
probe = run_pilot(client, MANIFEST, SAVE / 'probe_reflect_v1', modes=SIX_MODES, limit=1, max_calls=20)
print(probe)
probe_results = [json.loads(line) for line in (SAVE / 'probe_reflect_v1/results.jsonl').read_text().splitlines()]
print(json.dumps(probe_results, ensure_ascii=False, indent=2))
if probe['errors'] or probe['completed_cases'] != 6:
    raise RuntimeError('Проверка не прошла. Скачайте trace.jsonl из probe_reflect_v1 и разберите ошибку перед большим запуском.')

## Сравнение четырёх режимов

Шесть изображений × четыре режима = 24 случая; полный проход требует 60 вызовов. Лимит 70 оставляет небольшой запас на незавершённые случаи при переподключении. Все условия нейтральные: это проверка инфраструктуры и исходного поведения, ещё не эксперимент с ложными подсказками.

Первичный критерий: формат ответа и получение полного журнала. Научный критерий после независимой разметки: обнаружение двух допустимых образов с конкретными корректными признаками. На этой выборке нет отрицательных примеров, поэтому частоту выдуманных альтернатив оценивать нельзя.

При ошибках не удаляйте неудачные ответы и не подстраивайте промпты под них в той же папке запуска.

In [ ]:
if probe['errors'] or probe['completed_cases'] != 6:
    raise RuntimeError('Сначала нужна успешная проверка шести режимов в probe_reflect_v1.')
summary = run_pilot(client, MANIFEST, SAVE / 'pilot_reflect_v1', modes=SIX_MODES, max_calls=110)
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd

records = [json.loads(line) for line in (SAVE / 'pilot_reflect_v1/results.jsonl').read_text().splitlines()]
status_table = pd.DataFrame(records).groupby(['mode', 'status']).size().unstack(fill_value=0)
display(status_table)
print('Это технические статусы, НЕ точность модели.')

In [ ]:
if summary['completed_cases'] != summary['planned_cases']:
    raise RuntimeError('Сначала завершите все случаи пилота; увеличьте max_calls при необходимости.')
from types import SimpleNamespace
from illusion_agent.cli import export_review

REVIEW = SAVE / 'reviewer_a_reflect_v1'
if not REVIEW.exists():
    export_review(SimpleNamespace(run=str(SAVE / 'pilot_reflect_v1'), manifest=str(MANIFEST),
                                 output=str(REVIEW), seed=91))
print('Пакет для оценщика:', REVIEW)
print('Скопируйте ratings.csv двум оценщикам, оценки заполните независимо. Mapping из папки pilot_reflect_v1 им не передавайте.')

In [ ]:
import shutil

result_archive = shutil.make_archive('/content/bica_real_pilot_results', 'zip', SAVE)
files.download(result_archive)
print('Сохраните скачанный архив и передайте его для анализа. В нём нет API-ключей: этот запуск их не использует.')

## Версия v0.2: что именно проверяем

Новая архитектура использует независимую вторую интерпретацию (без истории и cue), а затем аудит гипотез с полной исходной картинкой. Режим `reflect_zoom` добавляет до двух фрагментов. Это не обученная с нуля VLM и не означает гарантированное улучшение.

Здесь **6 режимов**, 36 случаев и до 96 вызовов модели, считая все этапы. Исходные данные — 6 отладочных иллюзий AmbiBench, только нейтральный запрос, без согласованного двойного человеческого оценивания. Их нельзя использовать для доказательства превосходства.

Если вы изменили код, промпты или ревизию модели, используйте новую папку эксперимента: сохранённый конфиг не даст незаметно смешать условия. После исполнения сохраните `config.json`, `trace.jsonl`, `results.jsonl`, `summary.json`, скопированные картинки и `*_mapping.json` — последний файл не отдавать слепым оценщикам.

**Следующий обязательный этап:** дополнить датасет реальными обычными отрицательными примерами и истинными/ложными подсказками, размечать двумя людьми, затем сделать `score` и `paired-score`. Для официальных результатов AmbiBench/IllusionBench используйте исходные правила бенчмарка; текущая проверка ими не является.

Источники: https://huggingface.co/datasets/BLNL/AmbiBench, https://arxiv.org/abs/2501.00848. Подробности в `docs/06_self_reflection.md`.
